### Import all packages

In [ ]:
import matplotlib
import ikarus as iks
import ikarus.finite_elements
import ikarus.utils
import ikarus.assembler
import ikarus.dirichlet_values
import numpy as np
import scipy as sp
from scipy.optimize import minimize

import dune.grid
import dune.functions
from dune.vtk import vtkUnstructuredGridWriter, vtkWriter, RangeTypes, FieldInfo

from dune.vtk import  vtkWriter


### Create grid

In [ ]:
lowerLeft = [-1,-1]
upperRight = [1,1]
elements = [3,3]

grid = dune.grid.structuredGrid(lowerLeft,upperRight,elements)
grid.hierarchicalGrid.globalRefine(0)
grid.plot()

### Add Lagrangian basis

In [ ]:
basisLagrange1 = ikarus.basis(grid, dune.functions.Power(dune.functions.Lagrange(order=1), 2))
flatBasis = basisLagrange1.flat()
print('We have {} dofs.'.format(len(flatBasis)))
print('We have {} vertices.'.format(grid.size(2)))
print('We have {} elements.'.format(grid.size(0)))

### Initialize load factor

In [ ]:
lambdaLoad = iks.ValueWrapper(3.0)

### Define volume load and boundary loads

In [ ]:
def vL(x,lambdaVal) :
    return np.array([lambdaVal * x[0] * 2, 2 * lambdaVal * x[1] * 0])

def nL(x,lambdaVal) :
    return np.array([lambdaVal * 0, lambdaVal])

neumannVertices = np.zeros(grid.size(2), dtype=bool)
def loadTopEdgePredicate(x):
        return True if x[1] > 1-1e-8 else False

indexSet = grid.indexSet
for v in grid.vertices:
    neumannVertices[indexSet.index(v)]=loadTopEdgePredicate(v.geometry.center)

boundaryPatch = iks.utils.boundaryPatch(grid, neumannVertices)

volumeLoad = iks.finite_elements.volumeLoad2D(vL)
neumannLoad= iks.finite_elements.neumannBoundaryLoad(boundaryPatch, nL)

### Create material

In [ ]:
svk = iks.materials.StVenantKirchhoff(E=1000, nu=0.3)

svkPS = svk.asPlaneStress()

### Create vector of finite elements

In [ ]:
nonLinElastic = iks.finite_elements.nonLinearElastic(svkPS)

fes = []
for e in grid.elements:
    fes.append(iks.finite_elements.makeFE(basisLagrange1, nonLinElastic, volumeLoad, neumannLoad))
    fes[-1].bind(e)

### Create Dirichlet boundary conditions

In [ ]:
dirichletValues = iks.dirichletValues(flatBasis) 

def fixFirstIndex(vec, globalIndex):
        vec[0] = True

def fixAnotherVertex(vec, localIndex, localView):
    localView.index(localIndex)
    vec[1] = True

def fixLeftHandEdge(vec, localIndex, localView, intersection):
    if intersection.geometry.center[1] < -0.9:
        vec[localView.index(localIndex)] = True

dirichletValues.fixBoundaryDOFs(fixFirstIndex)
dirichletValues.fixBoundaryDOFsUsingLocalView(fixAnotherVertex)
dirichletValues.fixBoundaryDOFsUsingLocalViewAndIntersection(fixLeftHandEdge)

### Create assembler

In [ ]:
assembler = iks.assembler.sparseFlatAssembler(fes, dirichletValues)
dRed = np.zeros(assembler.reducedSize())

### Define functions

In [ ]:
def energy(dRedInput):
    reqL = ikarus.FERequirements()
    reqL.addAffordance(iks.ScalarAffordances.mechanicalPotentialEnergy)
    reqL.insertParameter(iks.FEParameter.loadfactor, lambdaLoad)

    dBig = assembler.createFullVector(dRedInput)
    reqL.insertGlobalSolution(iks.FESolutions.displacement, dBig)
    return assembler.getScalar(reqL)

def gradient(dRedInput):
    reqL = ikarus.FERequirements()
    reqL.addAffordance(iks.VectorAffordances.forces)
    reqL.insertParameter(iks.FEParameter.loadfactor, lambdaLoad)

    dBig = assembler.createFullVector(dRedInput)
    reqL.insertGlobalSolution(iks.FESolutions.displacement, dBig)
    return assembler.getReducedVector(reqL)

def hess(dRedInput):
    reqL = ikarus.FERequirements()
    reqL.addAffordance(iks.MatrixAffordances.stiffness)
    reqL.insertParameter(iks.FEParameter.loadfactor, lambdaLoad)

    dBig = assembler.createFullVector(dRedInput)
    reqL.insertGlobalSolution(iks.FESolutions.displacement, dBig)
    return assembler.getReducedMatrix(reqL).todense()

### Solve using scipy

In [ ]:
resultd = minimize(energy, x0=dRed, options={"disp": True}, tol=1e-14)
resultd2 = minimize(
    energy, x0=dRed, jac=gradient, options={"disp": True}, tol=1e-14
)
resultd3 = minimize(
    energy,
    method="trust-constr",
    x0=dRed,
    jac=gradient,
    hess=hess,
    options={"disp": True},
)
resultd4 = sp.optimize.root(gradient, jac=hess, x0=dRed, tol=1e-10)

In [ ]:
assert np.allclose(resultd.x, resultd2.x, atol=1e-6)
assert np.allclose(resultd3.x, resultd4.x)
assert np.all(abs(resultd3.grad) < 1e-8)
assert np.all(abs(resultd4.fun) < 1e-8)

In [ ]:
fullDisp = assembler.createFullVector(resultd2.x)

dispFunc = flatBasis.asFunction(fullDisp)

In [ ]:
req = ikarus.FERequirements()
req.insertGlobalSolution(iks.FESolutions.displacement, fullDisp)
res1 = fes[0].calculateAt(req, np.array([0.5, 0.5]), "PK2Stress")
print(res1)

### Plot here using matplotlib

In [ ]:
import dune.plotting
dune.plotting.plot(solution=dispFunc, gridLines=None)